In [1]:
"""
PIPELINE — Paso 2+3+4 — Tocantins.
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
import xarray as xr


# ====================================================================
# CONFIG
# ====================================================================

BASIN = "tocantins"
BASIN_FOLDER = "Tocantins"

EXPOSURE_ROOT = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/3.SPEI-12/Approach1/"

POP_TIF_100M = {
    "2015": f"{EXPOSURE_ROOT}2.Processing_outputs/worldpop_under18_2015_SA_100m.tif",
    "2030": f"{EXPOSURE_ROOT}2.Processing_outputs/worldpop_under18_2030_SA_100m.tif",
}

DROUGHT_BASE = f"/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.General_anual/2.Outputs/{BASIN_FOLDER}"

SCENARIOS = {
    "historical": {"drought_nc": f"{DROUGHT_BASE}/historical/drought_annual_{BASIN}_historical_ensemble.nc", "pop_year": "2015"},
    "ssp126":     {"drought_nc": f"{DROUGHT_BASE}/ssp126/drought_annual_{BASIN}_ssp126_ensemble.nc",         "pop_year": "2030"},
    "ssp585":     {"drought_nc": f"{DROUGHT_BASE}/ssp585/drought_annual_{BASIN}_ssp585_ensemble.nc",         "pop_year": "2030"},
}

POP_OUT_DIR = f"{EXPOSURE_ROOT}3.Population_regridded/{BASIN_FOLDER}/"
EXPOSURE_OUT_DIR = f"{EXPOSURE_ROOT}4.Final_outputs/{BASIN_FOLDER}/"
os.makedirs(POP_OUT_DIR, exist_ok=True)
os.makedirs(EXPOSURE_OUT_DIR, exist_ok=True)

POP_NODATA_FALLBACK = -99999.0
NODATA = -9999.0
BLOCK = 4096
VARS_SEQUIA = ["frequency", "duration", "severity"]
FORCE_RECALC_POP = True  # <- forzado a True para esta corrida: aseguramos que se recalcula, no reutiliza archivo viejo


# ====================================================================
# FUNCIONES
# ====================================================================

def get_target_grid(nc_path):
    ds = xr.open_dataset(nc_path)
    lat = ds["lat"].values.astype("float64")
    lon = ds["lon"].values.astype("float64")
    ds.close()
    res_lat = float(np.round(np.abs(np.diff(lat)).mean(), 8))
    res_lon = float(np.round(np.abs(np.diff(lon)).mean(), 8))
    west = float(lon.min() - res_lon / 2)
    north = float(lat.max() + res_lat / 2)
    n_rows, n_cols = len(lat), len(lon)
    transform = from_origin(west, north, res_lon, res_lat)
    return transform, (n_rows, n_cols), lat, lon, res_lat, res_lon


def regrid_population_blocksum(pop_tif_path, target_transform, target_shape, block=BLOCK):
    n_rows, n_cols = target_shape
    dst_data = np.zeros((n_rows, n_cols), dtype="float64")

    with rasterio.open(pop_tif_path) as src:
        src_nodata = src.nodata if src.nodata is not None else POP_NODATA_FALLBACK
        src_transform = src.transform

        t_west, t_north = target_transform.c, target_transform.f
        t_res_x, t_res_y = target_transform.a, -target_transform.e
        t_east = t_west + n_cols * t_res_x
        t_south = t_north - n_rows * t_res_y

        full_window = rasterio.windows.from_bounds(t_west, t_south, t_east, t_north, transform=src_transform)
        full_window = full_window.round_offsets().round_lengths()
        full_window = full_window.intersection(Window(0, 0, src.width, src.height))

        row_off0, col_off0 = int(full_window.row_off), int(full_window.col_off)
        n_rows_w, n_cols_w = int(full_window.height), int(full_window.width)

        for r0 in range(0, n_rows_w, block):
            h = min(block, n_rows_w - r0)
            for c0 in range(0, n_cols_w, block):
                w = min(block, n_cols_w - c0)
                win = Window(col_off0 + c0, row_off0 + r0, w, h)
                data = src.read(1, window=win)
                win_transform = src.window_transform(win)

                valid = data != src_nodata
                if not valid.any():
                    continue

                rows_idx, cols_idx = np.nonzero(valid)
                values = data[rows_idx, cols_idx].astype("float64")
                xs = win_transform.c + (cols_idx + 0.5) * win_transform.a
                ys = win_transform.f + (rows_idx + 0.5) * win_transform.e
                dst_col = np.floor((xs - t_west) / t_res_x).astype("int64")
                dst_row = np.floor((t_north - ys) / t_res_y).astype("int64")

                in_bounds = (dst_col >= 0) & (dst_col < n_cols) & (dst_row >= 0) & (dst_row < n_rows)
                if not in_bounds.any():
                    continue

                dst_col, dst_row = dst_col[in_bounds], dst_row[in_bounds]
                values_in = values[in_bounds]
                flat_idx = dst_row * n_cols + dst_col
                sums = np.bincount(flat_idx, weights=values_in, minlength=n_rows * n_cols)
                dst_data += sums.reshape(n_rows, n_cols)

    return dst_data.astype("float32")


def sum_source_in_bbox_chunked(pop_tif_path, west, south, east, north, block=BLOCK):
    total = 0.0
    with rasterio.open(pop_tif_path) as src:
        nodata = src.nodata if src.nodata is not None else POP_NODATA_FALLBACK
        full_window = rasterio.windows.from_bounds(west, south, east, north, transform=src.transform)
        full_window = full_window.round_offsets().round_lengths()
        full_window = full_window.intersection(Window(0, 0, src.width, src.height))
        row_off0, col_off0 = int(full_window.row_off), int(full_window.col_off)
        n_rows_w, n_cols_w = int(full_window.height), int(full_window.width)

        for r0 in range(0, n_rows_w, block):
            h = min(block, n_rows_w - r0)
            for c0 in range(0, n_cols_w, block):
                w = min(block, n_cols_w - c0)
                win = Window(col_off0 + c0, row_off0 + r0, w, h)
                data = src.read(1, window=win)
                valid = data != nodata
                total += float(data[valid].sum())
    return total


def save_population_nc(dst_data, lat_target, lon_target, out_path, year, basin):
    dst_data_ascending = dst_data[::-1, :]
    da = xr.DataArray(
        dst_data_ascending, dims=("lat", "lon"),
        coords={"lat": lat_target, "lon": lon_target},
        name="population_under18",
        attrs={
            "long_name": f"Population under 18 ({year}) resampled to drought grid",
            "units": "children per pixel",
            "source": "WorldPop, aggregated by sum (block-wise custom resampling)",
            "_FillValue": NODATA,
            "basin": basin,
        },
    )
    ds_out = da.to_dataset()
    ds_out.attrs["description"] = f"Static population <18, year {year}, regridded to {basin} drought grid"
    ds_out.to_netcdf(out_path)


def get_or_create_population(pop_year, target_transform, target_shape, lat_target, lon_target,
                              west, south, east, north):
    out_path = f"{POP_OUT_DIR}worldpop_under18_{pop_year}_{BASIN}.nc"

    if os.path.exists(out_path) and not FORCE_RECALC_POP:
        print(f"  Poblacion {pop_year}: ya existe, reutilizando -> {out_path}")
        return out_path

    print(f"  Poblacion {pop_year}: calculando...")
    dst_data = regrid_population_blocksum(POP_TIF_100M[pop_year], target_transform, target_shape)
    suma_post = float(dst_data.sum())
    suma_ref = sum_source_in_bbox_chunked(POP_TIF_100M[pop_year], west, south, east, north)
    diff_pct = 100 * (suma_post - suma_ref) / suma_ref if suma_ref != 0 else float("nan")
    print(f"    Suma regrid: {suma_post:,.1f} | Suma ref: {suma_ref:,.1f} | Diff: {diff_pct:.4f}%")
    if abs(diff_pct) > 0.5:
        print("    ATENCION: diferencia de conservacion de masa > 0.5% -- revisar.")

    save_population_nc(dst_data, lat_target, lon_target, out_path, pop_year, BASIN)
    print(f"    Guardado: {out_path}")
    return out_path


def calcular_exposicion(drought_nc, pop_nc, scenario, pop_year):
    ds_drought = xr.open_dataset(drought_nc)
    ds_pop = xr.open_dataset(pop_nc)

    lat_ok = np.array_equal(ds_drought["lat"].values, ds_pop["lat"].values)
    lon_ok = np.array_equal(ds_drought["lon"].values, ds_pop["lon"].values)
    if not (lat_ok and lon_ok):
        raise ValueError(f"[{scenario}] Grillas de sequia y poblacion NO coinciden exactamente.")

    pop_da = ds_pop["population_under18"]
    pop_valid = pop_da.notnull()

    exposure_vars = {}
    for var in VARS_SEQUIA:
        drought_da = ds_drought[var]

        min_val = float(drought_da.min(skipna=True))
        if min_val < 0:
            print(f"    ATENCION [{scenario}/{var}]: valores negativos detectados (min={min_val:.3f}).")

        drought_valid = drought_da.notnull()
        valid_mask = drought_valid & pop_valid
        exposure = xr.where(valid_mask, drought_da * pop_da, np.nan)
        exposure.name = f"exposure_{var}"
        exposure.attrs = {
            "long_name": f"Children under 18 exposed, weighted by {var}",
            "formula": f"{var}(pixel,year) x population_under18(pixel)",
            "units": f"children x {var}_units",
            "basin": BASIN, "scenario": scenario, "model": "ensemble", "population_year": pop_year,
        }
        exposure_vars[f"exposure_{var}"] = exposure

        n_valid, n_total = int(valid_mask.sum()), valid_mask.size
        print(f"    exposure_{var}: {n_valid}/{n_total} pixeles-año validos ({100*n_valid/n_total:.1f}%)")

    ds_exposure = xr.Dataset(exposure_vars, coords={
        "year": ds_drought["year"], "lat": ds_drought["lat"], "lon": ds_drought["lon"],
    })
    ds_exposure.attrs = {
        "description": f"Annual child drought exposure — {BASIN}, {scenario}, ensemble",
        "basin": BASIN, "scenario": scenario, "model": "ensemble",
        "population_source": f"WorldPop {pop_year} (static)",
    }

    out_nc = f"{EXPOSURE_OUT_DIR}exposure_annual_{BASIN}_{scenario}_ensemble.nc"
    encoding = {v: {"_FillValue": NODATA} for v in exposure_vars.keys()}
    ds_exposure.to_netcdf(out_nc, encoding=encoding)
    print(f"    Guardado: {out_nc}")

    years = ds_drought["year"].values
    resumen = {"year": years}
    for var in VARS_SEQUIA:
        serie = ds_exposure[f"exposure_{var}"].sum(dim=["lat", "lon"], skipna=True).values
        resumen[f"exposure_{var}"] = serie
        print(f"    exposure_{var}: promedio={np.nanmean(serie):,.1f} | "
              f"max={np.nanmax(serie):,.1f} (año {years[np.nanargmax(serie)]}) | "
              f"min={np.nanmin(serie):,.1f} (año {years[np.nanargmin(serie)]})")

    df_resumen = pd.DataFrame(resumen)
    csv_path = f"{EXPOSURE_OUT_DIR}resumen_anual_{BASIN}_{scenario}_ensemble.csv"
    df_resumen.to_csv(csv_path, index=False)
    print(f"    CSV resumen: {csv_path}")

    return out_nc, csv_path


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":
    print(f"{'='*70}\nPIPELINE — {BASIN}\n{'='*70}")

    print("\n--- PASO 2+3: Poblacion ---")
    target_transform, target_shape, lat_target, lon_target, res_lat, res_lon = get_target_grid(
        SCENARIOS["historical"]["drought_nc"]
    )
    n_rows, n_cols = target_shape
    west, north = target_transform.c, target_transform.f
    east = west + n_cols * res_lon
    south = north - n_rows * res_lat
    print(f"Grilla objetivo (desde historical): shape={target_shape}, "
          f"bbox=({west:.4f},{south:.4f},{east:.4f},{north:.4f})")

    for scenario, cfg in SCENARIOS.items():
        t2, s2, lat2, lon2, _, _ = get_target_grid(cfg["drought_nc"])
        same = np.array_equal(lat_target, lat2) and np.array_equal(lon_target, lon2)
        print(f"  Grilla {scenario} == grilla historical: {same}")
        if not same:
            raise ValueError(f"La grilla de {scenario} NO coincide con la de historical.")

    pop_nc_paths = {}
    for pop_year in ["2015", "2030"]:
        pop_nc_paths[pop_year] = get_or_create_population(
            pop_year, target_transform, target_shape, lat_target, lon_target, west, south, east, north
        )

    print("\n--- PASO 4: Exposicion ---")
    outputs = []
    for scenario, cfg in SCENARIOS.items():
        print(f"\n  Escenario: {scenario} (poblacion {cfg['pop_year']})")
        out_nc, csv_path = calcular_exposicion(
            cfg["drought_nc"], pop_nc_paths[cfg["pop_year"]], scenario, cfg["pop_year"]
        )
        outputs.append((scenario, out_nc, csv_path))

    print(f"\n\n{'='*70}\nRESUMEN — {BASIN}\n{'='*70}")
    print("Poblacion regrideada:")
    for y, p in pop_nc_paths.items():
        print(f"  {y}: {p}")
    print("\nExposicion por escenario:")
    for scenario, out_nc, csv_path in outputs:
        print(f"  {scenario}:\n    {out_nc}\n    {csv_path}")

PIPELINE — tocantins

--- PASO 2+3: Poblacion ---
Grilla objetivo (desde historical): shape=(34, 32), bbox=(-55.3333,-18.0000,-39.3333,-1.0000)
  Grilla historical == grilla historical: True
  Grilla ssp126 == grilla historical: True
  Grilla ssp585 == grilla historical: True
  Poblacion 2015: calculando...
    Suma regrid: 13,512,038.0 | Suma ref: 13,512,037.3 | Diff: 0.0000%
    Guardado: /data/brussel/vo/000/bvo00033/vsc11346/Exposure/3.SPEI-12/Approach1/3.Population_regridded/Tocantins/worldpop_under18_2015_tocantins.nc
  Poblacion 2030: calculando...
    Suma regrid: 11,771,828.0 | Suma ref: 11,771,827.6 | Diff: 0.0000%
    Guardado: /data/brussel/vo/000/bvo00033/vsc11346/Exposure/3.SPEI-12/Approach1/3.Population_regridded/Tocantins/worldpop_under18_2030_tocantins.nc

--- PASO 4: Exposicion ---

  Escenario: historical (poblacion 2015)
    exposure_frequency: 16980/32640 pixeles-año validos (52.0%)
    exposure_duration: 16980/32640 pixeles-año validos (52.0%)
    exposure_severit

In [2]:
"""
VERIFICACION — Tocantins. Poblacion regrideada + exposicion final.
"""

import xarray as xr
import pandas as pd
import numpy as np

BASIN = "tocantins"
BASIN_FOLDER = "Tocantins"

EXPOSURE_ROOT = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/3.SPEI-12/Approach1/"
POP_DIR = f"{EXPOSURE_ROOT}3.Population_regridded/{BASIN_FOLDER}/"
EXPOSURE_DIR = f"{EXPOSURE_ROOT}4.Final_outputs/{BASIN_FOLDER}/"

VARS_SEQUIA = ["frequency", "duration", "severity"]
ESCENARIOS = ["historical", "ssp126", "ssp585"]


print(f"\n{'#'*70}\nPARTE 1 — POBLACION REGRIDEADA — {BASIN}\n{'#'*70}")

archivos_pop = {
    "2015": POP_DIR + f"worldpop_under18_2015_{BASIN}.nc",
    "2030": POP_DIR + f"worldpop_under18_2030_{BASIN}.nc",
}

datos_pop = {}
for year, path in archivos_pop.items():
    print(f"\n{'='*60}\n{year}\n{'='*60}")
    ds = xr.open_dataset(path)
    da = ds["population_under18"]
    print(f"Shape: {da.shape}")
    print(f"Lat range: {float(da.lat.min()):.4f} a {float(da.lat.max()):.4f}")
    print(f"Lon range: {float(da.lon.min()):.4f} a {float(da.lon.max()):.4f}")

    valores = da.values
    n_validos = int(np.isfinite(valores).sum())
    n_total = valores.size
    print(f"Píxeles válidos: {n_validos} de {n_total} ({100*n_validos/n_total:.1f}%)")
    print(f"Min: {np.nanmin(valores):.4f}  |  Max: {np.nanmax(valores):.4f}")
    print(f"Suma total (población <18 en la cuenca): {np.nansum(valores):,.1f}")

    n_negativos = int((valores[np.isfinite(valores)] < 0).sum())
    print(f"Píxeles negativos: {n_negativos}")

    datos_pop[year] = {"lat": da.lat.values, "lon": da.lon.values, "sum": np.nansum(valores),
                        "mask_valida": np.isfinite(valores)}
    ds.close()

print(f"\n{'='*60}\nConsistencia 2015 vs 2030\n{'='*60}")
print(f"Misma lat: {np.array_equal(datos_pop['2015']['lat'], datos_pop['2030']['lat'])}")
print(f"Misma lon: {np.array_equal(datos_pop['2015']['lon'], datos_pop['2030']['lon'])}")
print(f"Misma máscara válida: {np.array_equal(datos_pop['2015']['mask_valida'], datos_pop['2030']['mask_valida'])}")
caida_pct = 100 * (datos_pop['2030']['sum'] - datos_pop['2015']['sum']) / datos_pop['2015']['sum']
print(f"Cambio en población <18 (2015->2030): {caida_pct:+.2f}%")


print(f"\n\n{'#'*70}\nPARTE 2 — EXPOSICION FINAL — {BASIN}\n{'#'*70}")

promedios = {var: {} for var in VARS_SEQUIA}

for scenario in ESCENARIOS:
    print(f"\n{'='*70}\n{scenario}\n{'='*70}")

    nc_path = EXPOSURE_DIR + f"exposure_annual_{BASIN}_{scenario}_ensemble.nc"
    ds = xr.open_dataset(nc_path)
    print(f"Años: {int(ds.year.min())}–{int(ds.year.max())} (n={ds.year.size})")

    for var in VARS_SEQUIA:
        da = ds[f"exposure_{var}"]
        valores = da.values
        n_validos = int(np.isfinite(valores).sum())
        n_total = valores.size
        n_negativos = int((valores[np.isfinite(valores)] < 0).sum())

        print(f"\n  exposure_{var}:")
        print(f"    Píxeles-año válidos: {n_validos}/{n_total} ({100*n_validos/n_total:.1f}%)")
        print(f"    Min: {np.nanmin(valores):,.2f}  |  Max: {np.nanmax(valores):,.2f}")
        print(f"    Negativos: {n_negativos}" + ("  <-- REVISAR" if n_negativos > 0 else ""))

    csv_path = EXPOSURE_DIR + f"resumen_anual_{BASIN}_{scenario}_ensemble.csv"
    df = pd.read_csv(csv_path)

    for var in VARS_SEQUIA:
        serie_nc = ds[f"exposure_{var}"].sum(dim=["lat", "lon"], skipna=True).values
        serie_csv = df[f"exposure_{var}"].values
        coincide = np.allclose(serie_nc, serie_csv, rtol=1e-4, equal_nan=True)
        print(f"    exposure_{var}: NetCDF y CSV coinciden -> {coincide}")
        promedios[var][scenario] = df[f"exposure_{var}"].mean()

    ds.close()

print(f"\n{'='*70}\nComparación entre escenarios (promedio de la serie completa)\n{'='*70}")
for var in VARS_SEQUIA:
    print(f"\n  exposure_{var}:")
    for scenario in ESCENARIOS:
        print(f"    {scenario}: promedio={promedios[var][scenario]:,.1f}")


######################################################################
PARTE 1 — POBLACION REGRIDEADA — tocantins
######################################################################

2015
Shape: (34, 32)
Lat range: -17.7500 a -1.2500
Lon range: -55.0833 a -39.5833
Píxeles válidos: 1088 de 1088 (100.0%)
Min: 0.0000  |  Max: 577649.0000
Suma total (población <18 en la cuenca): 13,512,038.0
Píxeles negativos: 0

2030
Shape: (34, 32)
Lat range: -17.7500 a -1.2500
Lon range: -55.0833 a -39.5833
Píxeles válidos: 1088 de 1088 (100.0%)
Min: 0.0000  |  Max: 569364.4375
Suma total (población <18 en la cuenca): 11,771,827.0
Píxeles negativos: 0

Consistencia 2015 vs 2030
Misma lat: True
Misma lon: True
Misma máscara válida: True
Cambio en población <18 (2015->2030): -12.88%


######################################################################
PARTE 2 — EXPOSICION FINAL — tocantins
######################################################################

historical
Años: 1985–2014 (n=30)

  e